In [1]:
import numpy as np
from pyscf import gto, scf, lo, mp, cc
mol = gto.Mole()
mol.verbose = 4
mol.atom = '''
O   -1.485163346097   -0.114724564047    0.000000000000
H   -1.868415346097    0.762298435953    0.000000000000
H   -0.533833346097    0.040507435953    0.000000000000
O    1.416468653903    0.111264435953    0.000000000000
H    1.746241653903   -0.373945564047   -0.758561000000
H    1.746241653903   -0.373945564047    0.758561000000
'''
mol.basis = 'cc-pvdz'
mol.precision = 1e-10
mol.build()
mf = scf.RHF(mol).density_fit()
mf.kernel()

frozen = 2
mymp = mp.MP2(mf, frozen=frozen)
mymp.kernel()
efull_mp2 = mymp.e_corr
print(f'MP2 Corr = {efull_mp2:.8f}')

mycc = cc.CCSD(mf, frozen=frozen)
mycc.kernel()
efull_ccsd = mycc.e_corr
print(f'CCSD Corr = {efull_ccsd:.8f}')

efull_t = mycc.ccsd_t()
efull_ccsd_t = efull_ccsd + efull_t
print(f'CCSD(T) Corr = {efull_ccsd_t:.8f}')

System: uname_result(system='Linux', node='sharmagroup-rn', release='6.17.0-23-generic', version='#23~24.04.1-Ubuntu SMP PREEMPT_DYNAMIC Tue Apr 14 16:11:48 UTC 2', machine='x86_64')  Threads 16
Python 3.12.13 | packaged by Anaconda, Inc. | (main, Mar 19 2026, 20:20:58) [GCC 14.3.0]
numpy 2.4.4  scipy 1.17.1  h5py 3.16.0
Date: Tue May  5 21:11:26 2026
PySCF version 2.12.1
PySCF path  /home/sharmagroup/sharmagroup/pyscf
GIT ORIG_HEAD 3d1768f5e33b144b606c3d2c81c12ee54d794501
GIT HEAD (branch master) f0861da51f017364d8bbaa20b742a94f3733305f

[ENV] OLD_PYSCF_EXT_PATH /home/sharmagroup/sharmagroup/pyscf-forge:
[ENV] PYSCF_EXT_PATH /home/sharmagroup/sharmagroup/pyscf-forge:/home/sharmagroup/sharmagroup/pyscf-forge:
[CONFIG] conf_file None
[INPUT] verbose = 4
[INPUT] num. atoms = 6
[INPUT] num. electrons = 20
[INPUT] charge = 0
[INPUT] spin (= nelec alpha-beta = 2S) = 0
[INPUT] symmetry False subgroup None
[INPUT] Mole.unit = angstrom
[INPUT] Symbol           X                Y               

In [3]:
print(f'MP2 Corr = {efull_mp2:.8f}')
print(f'CCSD Corr = {efull_ccsd:.8f}')
print(f'CCSD(T) Corr = {efull_ccsd_t:.8f}')

MP2 Corr = -0.40609899
CCSD Corr = -0.42461940
CCSD(T) Corr = -0.43105819


In [27]:
mol.aoslice_by_atom()

array([[ 0,  5,  0, 14],
       [ 5,  8, 14, 19],
       [ 8, 11, 19, 24],
       [11, 16, 24, 38],
       [16, 19, 38, 43],
       [19, 22, 43, 48]])

In [ ]:
aoslices = mol.aoslice_by_atom()

for ia in range(mol.natm):
    sh0, sh1, p0, p1 = aoslices[ia]
    symb = mol.atom_symbol(ia)        # e.g. 'O', 'H1', 'H@2' (keeps any user labels)
    # elem = mol.atom_pure_symbol(ia)   # e.g. 'O', 'H'        (just the element)
    # Z    = mol.atom_charge(ia)        # nuclear charge
    # xyz  = mol.atom_coord(ia)         # coordinates in Bohr
    print(f"Atom {ia} ({symb})")

Atom 0 (O) O
Atom 1 (H) H
Atom 2 (H) H
Atom 3 (O) O
Atom 4 (H) H
Atom 5 (H) H


In [18]:
from pyscf.lno import LNOCCSD, LNOCCSD_T
from pyscf.lno.tools import autofrag_iao

# def test_lno_iao_by_thresh(self):
mol = mycc.mol
mf = mycc._scf
frozen = mycc.frozen

# IAO localization
orbocc = mf.mo_coeff[:,frozen:np.count_nonzero(mf.mo_occ)]
lo_coeff = lo.iao.iao(mol, orbocc)
lo_coeff = lo.orth.vec_lowdin(lo_coeff, mf.get_ovlp())
moliao = lo.iao.reference_mol(mol)

frag_lolist = autofrag_iao(moliao)

gamma = 10
threshs = [3e-5]
# refs = [
#     [-0.4054784012,-0.4240686326,-0.4303996712],
#     [-0.4060479828,-0.4245745223,-0.4309965749],
# ]

emp2_iao = np.zeros(len(threshs))
eccsd_iao = np.zeros(len(threshs))
eccsd_t_iao = np.zeros(len(threshs))

for i, thresh in enumerate(threshs):
    mcc = LNOCCSD(mf, lo_coeff, frag_lolist, frozen=frozen).set(verbose=5)
    mcc.lno_thresh = [thresh*gamma,thresh]
    mcc.kernel()
    emp2_iao[i] = mcc.e_corr_pt2
    eccsd_iao[i] = mcc.e_corr_ccsd
    eccsd_t_iao[i] = mcc.e_corr_ccsd_t


******** <class 'pyscf.lno.lnoccsd.LNOCCSD'> ********
nocc = 8, nmo = 46
frozen orbitals 2
max_memory 4000 MB (current use 371 MB)
nfrag = 6  nlo = 14
frag_lolist = [[0, 1, 2, 3, 4], [5], [6], [7, 8, 9, 10, 11], [12], [13]]
frag_wghtlist = None
lno_type = ['1h', '1h']
lno_thresh = [0.00030000000000000003, 3e-05]
lno_pct_occ = None
lno_norb = None
lo_proj_thresh = 1e-10
lo_proj_thresh_active = 0.1
verbose_imp = 2
_ovL = None
_ovL_to_save = None
force_outcore_ao2mo = False
_match_oldcode = False
_max_las_size_ccsd = 1000
_max_las_size_ccsd_t = 1000
Regularized frag_wghtlist = [1. 1. 1. 1. 1. 1.]
    CPU time for LO and fragment        0.00 sec, wall time      0.00 sec

WARN: Input vhf is not found. Building vhf from SCF MO.

ao2mo est mem= 0.54 MB  avail mem= 3628.97 MB
aux blksize for incore ao2mo: 232/232
    CPU time for Integral xform         0.02 sec, wall time      0.00 sec
LO occ proj: 4 active | 1 standby | 3 orthogonal
LO vir proj: 2 active | 3 standby | 33 orthogonal
    CPU t

In [55]:
moliao.aoslice_by_atom()
aoslices = moliao.aoslice_by_atom()
for ia in range(moliao.natm):
    sh0, sh1, p0, p1 = aoslices[ia]
    symb = moliao.atom_symbol(ia)        # e.g. 'O', 'H1', 'H@2' (keeps any user labels)
    elem = moliao.atom_pure_symbol(ia)   # e.g. 'O', 'H'        (just the element)
    Z    = moliao.atom_charge(ia)        # nuclear charge
    xyz  = moliao.atom_coord(ia)         # coordinates in Bohr
    print(f"Atom {ia} ({symb}) {Z} {xyz} AOs {p0}:{p1}")

Atom 0 (O) 8 [-2.80655197 -0.21679801  0.        ] AOs 0:5
Atom 1 (H) 1 [-3.53079329  1.44053527  0.        ] AOs 5:6
Atom 2 (H) 1 [-1.00879882  0.07654796  0.        ] AOs 6:7
Atom 3 (O) 8 [2.67673782 0.21025931 0.        ] AOs 7:12
Atom 4 (H) 1 [ 3.29991847 -0.7066547  -1.43347254] AOs 12:13
Atom 5 (H) 1 [ 3.29991847 -0.7066547   1.43347254] AOs 13:14


In [62]:
mol.elements

['O', 'H', 'H', 'O', 'H', 'H']

In [37]:
from pyscf.tools import molden

with open('water_dimer_iao.molden', 'w') as f:
    molden.header(mol, f)
    molden.orbital_coeff(mol, f, lo_coeff)

In [3]:
from collections.abc import Iterable
from pyscf.lno import lnoccsd
from afqmc.lno_afqmc.lno_afqmc import lno_ccsd
from pyscf.lno.tools import autofrag_iao
from jax import random
import os

In [ ]:
options = {'n_eql': 3,
           'n_prop_steps': 50,
           'n_ene_blocks': 1,
           'n_sr_blocks': 5,
           'n_blocks': 100,
           'n_walkers': 100,
           'n_batch': 1,
           'seed': 17,
           'walker_type': 'rhf',
           'trial': 'ccsd_pt2',
           'dt':0.005,
           'use_gpu': True,
           'max_error': 1e-3
           }

# import jax
# jax.config.update("jax_enable_x64", True)

mol = mycc.mol
mf = mycc._scf
frozen = mycc.frozen

# IAO localization
orbocc = mf.mo_coeff[:,frozen:np.count_nonzero(mf.mo_occ)]
lo_coeff_iao = lo.iao.iao(mol, orbocc)
lo_coeff_iao = lo.orth.vec_lowdin(lo_coeff_iao, mf.get_ovlp())
moliao = lo.iao.reference_mol(mol)

frag_lolist = autofrag_iao(moliao)